# Đề bài về nhà


## Yêu cầu


- Tự viết code cho mô hình Linear Regression theo công thức đã được dạy trong buổi lý thuyết trên lớp.
- Tự viết hàm dự đoán.
- Huấn luyện cả mô hình của thư viện và mô hình mình tự viết.
- In ra các trọng số: w0, w1, w2, ..., wn của cả 2 mô hình đã huấn luyện để quan sát và so sánh.
- Dự đoán dữ liệu tập test bằng cả 2 mô hình (mô hình thư viện thì dùng hàm predict() của thư viện, mô hình tự viết dùng hàm dự đoán tự viết), in ra kết quả bằng Dataframe như trong bài thực hành trên lớp.
- Tính RMSE trên tập test cho cả 2 mô hình và so sánh.


## Dữ liệu


Tập dữ liệu giá nhà ở Boston đã có sẵn trên sklearn, dữ liệu đã được chuẩn hóa và chia thành tập train, tập test


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math

from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score

# Đọc dữ liệu

Dữ liệu về giá nhà ở Boston được hỗ trợ bởi sklearn, đọc dữ liệu thông qua hàm `datasets.load_boston()`

Xem thêm các bộ dữ liệu khác tại https://scikit-learn.org/stable/datasets/index.html#toy-datasets.

Dữ liệu được chia thành các thành phần data và target như tập diabetes. Dữ liệu cũng đã được chuẩn hóa, chỉ cần gọi ra và huấn luyện


In [2]:
# Khởi tạo đối tượng giả lập để tương thích ngược với cấu trúc dataset.data và dataset.target
class BostonDataset:
    def __init__(self, data, target):
        self.data = data
        self.target = target


# Đọc dữ liệu từ file BostonDataset.txt cục bộ (do datasets.load_boston() đã bị gỡ bỏ ở scikit-learn mới)
import os

file_path = "BostonDataset.txt"
if not os.path.exists(file_path):
    file_path = os.path.join("3-Regression", "Homework", "BostonDataset.txt")

raw_df = pd.read_csv(file_path, sep="\\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]
dataset = BostonDataset(data, target)

print("Số chiều dữ liệu input: ", dataset.data.shape)
print("Số chiều dữ liệu target: ", dataset.target.shape)
print()

print("5 mẫu dữ liệu đầu tiên:")
print("input: ", dataset.data[:5])
print("target: ", dataset.target[:5])

Số chiều dữ liệu input:  (506, 13)
Số chiều dữ liệu target:  (506,)

5 mẫu dữ liệu đầu tiên:
input:  [[6.3200e-03 1.8000e+01 2.3100e+00 0.0000e+00 5.3800e-01 6.5750e+00
  6.5200e+01 4.0900e+00 1.0000e+00 2.9600e+02 1.5300e+01 3.9690e+02
  4.9800e+00]
 [2.7310e-02 0.0000e+00 7.0700e+00 0.0000e+00 4.6900e-01 6.4210e+00
  7.8900e+01 4.9671e+00 2.0000e+00 2.4200e+02 1.7800e+01 3.9690e+02
  9.1400e+00]
 [2.7290e-02 0.0000e+00 7.0700e+00 0.0000e+00 4.6900e-01 7.1850e+00
  6.1100e+01 4.9671e+00 2.0000e+00 2.4200e+02 1.7800e+01 3.9283e+02
  4.0300e+00]
 [3.2370e-02 0.0000e+00 2.1800e+00 0.0000e+00 4.5800e-01 6.9980e+00
  4.5800e+01 6.0622e+00 3.0000e+00 2.2200e+02 1.8700e+01 3.9463e+02
  2.9400e+00]
 [6.9050e-02 0.0000e+00 2.1800e+00 0.0000e+00 4.5800e-01 7.1470e+00
  5.4200e+01 6.0622e+00 3.0000e+00 2.2200e+02 1.8700e+01 3.9690e+02
  5.3300e+00]]
target:  [24.  21.6 34.7 33.4 36.2]


**Chia dữ liệu làm 2 phần training 362 mẫu và testing 80 mẫu**


In [3]:
# cat nho du lieu, lay 1 phan cho qua trinh thu nghiem,
# chia train test cac mau du lieu
# dataset_X = dataset.data[:, np.newaxis, 2]
dataset_X = dataset.data

dataset_X_train = dataset_X[:404]
dataset_y_train = dataset.target[:404]

dataset_X_test = dataset_X[405:]
dataset_y_test = dataset.target[405:]

# Xây dựng mô hình


## Xây dựng mô hình bằng thư viện


In [4]:
# Khởi tạo mô hình Linear Regression của thư viện scikit-learn
regr = linear_model.LinearRegression()

## Xây dựng mô hình Linear Regression tự viết


In [5]:
class MyLinearRegression:
    def __init__(self):
        self.w = None
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X, y):
        # Thêm cột bias (x0 = 1) vào đầu ma trận đặc trưng X
        X_b = np.hstack((np.ones((X.shape[0], 1)), X))

        # Tính nghiệm đóng sử dụng giả nghịch đảo để đảm bảo tính ổn định số học
        self.w = np.linalg.pinv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)

        # Lưu trữ intercept_ và coef_
        self.intercept_ = self.w[0]
        self.coef_ = self.w[1:]
        return self

    def predict(self, X):
        # Thêm cột bias (x0 = 1) trước khi dự đoán
        X_b = np.hstack((np.ones((X.shape[0], 1)), X))
        return X_b.dot(self.w)

## Hàm test mô hình tự viết


In [6]:
# Kiểm tra mô hình tự viết trên một tập dữ liệu nhỏ ngẫu nhiên
test_X = np.array([[1, 2], [3, 4], [5, 6]])
test_y = np.array([5, 11, 17])  # Quan hệ tuyến tính thực tế: y = 3*x1 + 1*x2 + 0
test_model = MyLinearRegression()
test_model.fit(test_X, test_y)
print("Bộ trọng số tự tính:", test_model.w)
print("Dự đoán mẫu mới [2, 3]:", test_model.predict(np.array([[2, 3]])))

Bộ trọng số tự tính: [0.33333333 1.33333333 1.66666667]
Dự đoán mẫu mới [2, 3]: [8.]


# Huấn luyện mô hình


## Huấn luyện mô hình của thư viện


In [7]:
# Huấn luyện mô hình của thư viện
regr.fit(dataset_X_train, dataset_y_train)
print("Sklearn Intercept (w0):", regr.intercept_)
print("Sklearn Coefficients (w1...wn):", regr.coef_)

Sklearn Intercept (w0): 30.077166922901817
Sklearn Coefficients (w1...wn): [-2.02135297e-01  4.41276341e-02  5.26739364e-02  1.88474315e+00
 -1.49281487e+01  4.76038673e+00  2.88734527e-03 -1.30025278e+00
  4.61661953e-01 -1.55434673e-02 -8.11632369e-01 -1.97174433e-03
 -5.32273431e-01]


## Training mô hình bằng Linear regression tự viết


In [8]:
# Huấn luyện mô hình tự viết
my_regr = MyLinearRegression()
my_regr.fit(dataset_X_train, dataset_y_train)
print("My Intercept (w0):", my_regr.intercept_)
print("My Coefficients (w1...wn):", my_regr.coef_)

My Intercept (w0): 30.07716692084397
My Coefficients (w1...wn): [-2.02135297e-01  4.41276341e-02  5.26739364e-02  1.88474315e+00
 -1.49281487e+01  4.76038673e+00  2.88734527e-03 -1.30025278e+00
  4.61661953e-01 -1.55434673e-02 -8.11632369e-01 -1.97174433e-03
 -5.32273431e-01]


# Dự đoán các mẫu dữ liệu


## Dự đoán các mẫu dữ liệu theo mô hình của thư viện


In [9]:
# Dự đoán trên tập kiểm thử bằng mô hình sklearn
y_pred_sk = regr.predict(dataset_X_test)

## Dự đoán các mẫu dữ liệu tính theo linear regression tự viết


In [10]:
# Dự đoán trên tập kiểm thử bằng mô hình tự viết và so sánh kết quả trong DataFrame
y_pred_my = my_regr.predict(dataset_X_test)

df_compare = pd.DataFrame(
    {"Actual": dataset_y_test, "Sklearn_Predict": y_pred_sk, "My_Predict": y_pred_my}
)
df_compare.head(20)

,Actual,Sklearn_Predict,My_Predict
0,5.0,3.787057,3.787057
1,11.9,6.640550,6.640550
2,27.9,21.312765,21.312765
3,17.2,15.412714,15.412714
4,27.5,23.652298,23.652298
5,15.0,16.585687,16.585687
6,17.2,22.239823,22.239823
7,17.9,4.597394,4.597394
8,16.3,12.316953,12.316953
9,7.0,-4.441254,-4.441254


## Đánh giá mô hình linear regression của thư viện


In [11]:
# Tính toán sai số RMSE trên tập kiểm thử đối với mô hình thư viện
rmse_sk = math.sqrt(mean_squared_error(dataset_y_test, y_pred_sk))
print("RMSE Sklearn: ", rmse_sk)

RMSE Sklearn:  5.749521870254049


## Đánh giá mô hình linear regression tự viết


In [12]:
# Tính toán sai số RMSE trên tập kiểm thử đối với mô hình tự viết và so sánh
rmse_my = math.sqrt(mean_squared_error(dataset_y_test, y_pred_my))
print("RMSE My Model: ", rmse_my)
print("Độ lệch RMSE giữa 2 mô hình:", abs(rmse_sk - rmse_my))

RMSE My Model:  5.749521870204724
Độ lệch RMSE giữa 2 mô hình: 4.9324988538046455e-11
